In [1]:
import numpy as np

from physics.systems.particles.box import Box
from physics.systems.particles.energy import Energy
from physics.solvers.runge_kutta import RungeKutta4
from physics.systems.particles.motion import Motion
from physics.visualization.video import Video


In [2]:
# Parameters
N, box_size = 20, 100.0
dt, t0, t_max, v_max = 0.0001, 0.0, 10.0, 25.0
m, q, g = 1.0, 50.0, -10.0
name = "particles"
playback_speed = 1.0
fps = 60

In [3]:
print("Initializing Simulation...")

np.random.seed(42)
initial_state = np.zeros((N, 4))
initial_state[:, 0:2] = np.random.uniform(box_size*0.1, box_size*0.9, (N, 2))
initial_state[:, 2:4] = np.random.uniform(-v_max, v_max, (N, 2))
physics_system = Motion(g, q, m, N)
box = Box(box_size)
energy = Energy(g, q, m, N)

solver = RungeKutta4(physics_system)
solver.set_initial_condition(initial_state)
solver.dt = dt

video = Video(name=name, t0=t0, t_max=t_max, dt=dt, playback_speed=playback_speed, fps=fps)
total_steps = int((t_max - t0) / dt)
history = np.zeros((video.num_saves, N, 4))

current_state = initial_state.copy()
current_t = t0
save_idx = 0
print(f"Solving the system with [{t0}, {t_max}] at {total_steps} steps")
for step in range(total_steps):
    # The observer dictates the sampling interval
    if step % video.save_interval == 0 and save_idx < video.num_saves:
        history[save_idx] = current_state     
        save_idx += 1
    current_state = solver.advance(current_t, current_state)    
    current_state = box.collide(current_state)
    current_t += dt
print("Simulation complete. Generating the video...")
video.create_from_particles(history, box_size)

Initializing Simulation...

--- 🎬 Video Synchronization Report ---
Simulation Setup : t=[0, 10.0] with dt=0.0001 (100000 total steps)
Target Speed     : 1.0x Real-time (10.00s @ 60 FPS)
Calculated Sync  : Set save_interval = 166
Actual Output    : 10.03 seconds (602 total frames)
---------------------------------------

Solving the system with [0.0, 10.0] at 100000 steps
Simulation complete. Generating the video...

[Video: particles] Exporting particle frames...

[Video: particles] Compiling MP4 with FFmpeg...
[Video: particles] Success! Video saved as 'animations/videos/particles.mp4'
